In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F

# Read execution manifest from Bronze
execution_manifest = spark.table(
    f"{CATALOGS['bronze']}.warframe_market.execution_manifest"
)

display(execution_manifest)

In [0]:
# Build execution dimension
dim_execution = (
    execution_manifest
    .select(
        "execution_id",
        "started_at",
        "finished_at",
        F.to_date("started_at").alias("execution_date"),
        "execution_duration_seconds",
        "status",
        "response_count",
        "item_count",
        "order_response_count",
        "total_orders",
        "source_file"
    )
)

display(dim_execution)

In [0]:
# Validate execution dimension
validation = dim_execution.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("execution_id").alias("unique_executions"),
    F.sum(
        F.when(F.col("execution_id").isNull(), 1).otherwise(0)
    ).alias("null_execution_ids"),
    F.sum(
        F.when(
            F.col("started_at").isNull()
            | F.col("finished_at").isNull(),
            1
        ).otherwise(0)
    ).alias("null_timestamps"),
    F.sum(
        F.when(F.col("finished_at") < F.col("started_at"), 1).otherwise(0)
    ).alias("invalid_time_ranges"),
    F.sum(
        F.when(F.col("total_orders") < 0, 1).otherwise(0)
    ).alias("negative_order_counts")
)

display(validation)

In [0]:
from delta.tables import DeltaTable

TARGET_TABLE = get_table("intermediate", "dim_execution")

if spark.catalog.tableExists(TARGET_TABLE):

    target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        target.alias("target")
        .merge(
            dim_execution.alias("source"),
            "target.execution_id = source.execution_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    (
        dim_execution.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )

print(f"Table successfully materialized: {TARGET_TABLE}")

In [0]:
# Validate persisted execution dimension
dim_execution_saved = spark.table(TARGET_TABLE)

print(f"Persisted records: {dim_execution_saved.count():,}")

display(
    dim_execution_saved
    .orderBy("started_at")
)